# SK 08 - Multi-Agent orchestration of ChatCompletion + Assistant + AI Foundry + OpenAI Response Agents
## using [YAML declarative specification](https://learn.microsoft.com/en-us/semantic-kernel/frameworks/agent/agent-types/azure-ai-agent?pivots=programming-language-python#declarative-spec)
Possible types accepting YAML specification:
- chat_completion_agent
- foundry_agent
- azure_assistant
- azure_responses
- openai_assistant
- openai_responses

# Constants and Libraries

In [1]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
import importlib.metadata

if not load_dotenv("./../config/credentials_my.env"):
    print("Environment variables not loaded, cell execution stopped")
else:
    print("Environment variables have been loaded ;-)")

agent_name = "sk_aifoundry_agent-chocolate-lines"

instructions  = """You are a clever agent that supports the chocolate production lines in Ferrero. You have full access to Internet. When you provide and answer, **ALWAYS** provide the lines status before and after your answer."""

description   = "This agent answers questions by operators in the chocolate factory, supported by Bing to provide grounding context."""

project_endpoint = os.environ["AIF_BAS_PROJECT_ENDPOINT"] # AIF_BAS_PROJECT_ENDPOINT or AIF_STD_PROJECT_ENDPOINT
deployment_name =  os.environ["MODEL_DEPLOYMENT_NAME"]
openai_api_version = os.environ["OPENAI_API_VERSION"] # not less than 2025-03-01-preview
openai_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]

credential = DefaultAzureCredential()

print(f'OpenAI Endpoint: {openai_endpoint}')
print(f'Project Endpoint: {project_endpoint}')
print(f'OpenAI API Version: {openai_api_version}')
print(f"azure-ai-projects library installed version: {importlib.metadata.version("azure-ai-projects")}")
print(f"azure-ai-agents library installed version: {importlib.metadata.version("azure-ai-agents")}")

Environment variables have been loaded ;-)
OpenAI Endpoint: https://aif1bassvj36b.cognitiveservices.azure.com/
Project Endpoint: https://aif1bassvj36b.services.ai.azure.com/api/projects/aif1basswcprj01
OpenAI API Version: 2025-04-01-preview
azure-ai-projects library installed version: 1.0.0
azure-ai-agents library installed version: 1.1.0b4


# Universal `ChatWithAgentStreamAsync`
The following function works with any SK agent (built from ChatCompletion / Assistant / Response / AI Foundry) implementing a streaming response

In [2]:
async def ChatWithAgentStreamAsync(agent, USER_INPUTS: list) -> None:
    # return
    from semantic_kernel.agents import AzureAIAgentThread
    from semantic_kernel.contents import AuthorRole
    
    i=0
    for user_input in USER_INPUTS:
        print(f"\n************************************\nMessage {i} from {AuthorRole.USER}: '{user_input}'")
        # Invoke the agent for the specified task
        is_code = False
        last_role = None
        async for response in agent.invoke_stream(
            messages=user_input,
        ):
            current_is_code = response.metadata.get("code", False)

            if current_is_code:
                if not is_code:
                    print("\n\n```python")
                    is_code = True
                print(response.content, end="", flush=True)
            else:
                if is_code:
                    print("\n```")
                    is_code = False
                    last_role = None
                if hasattr(response, "role") and response.role is not None and last_role != response.role:
                    print(f"\n# {response.role}: ", end="", flush=True)
                    last_role = response.role
                print(response.content, end="", flush=True)
        if is_code:
            print("```\n")
        print()

# 1. Chat Completion Agent - `emotionevoker_agent`

## Load the agent definition

In [3]:
# Read the agent template from the file
with open("./_agents/emotionevoker_agent.yaml", "r") as file:
    fstring_template = file.read()

# replace variables and fix carriage returns
emotionevoker_agent_specs = eval(f"f'''{fstring_template}'''")
print(emotionevoker_agent_specs)

type: chat_completion_agent
name: emotionevoker_agent
description: Creates a question designed to evoke a sensory or emotional association with a creature - not just any animal, but one that could inspire a Ferrero flavor, mascot, or campaign.
model:
  id: gpt-4o
  options:
    temperature: 0.4
instructions: >
  # ROLE
  The Muse Whisperer

  # YOUR OBJECTIVE
  - Craft a poetic, sensory-rich question that leads to an emotionally resonant animal.

  # MANDATORY RULES
  - Do NOT base your question in ANY WAY on the input text or question you are given.
  - Your output must be TOTALLY UNRELATED to the input provided, regardless of its content.
  - Ignore the context or any associations implied by the input.

  # INSPIRATION
  Refer to the following examples to craft your question:
  - Which creature feels like velvet and tastes like nostalgia?
  - What animal embodies the joy of unwrapping a secret?
  - Which beast could inspire a hazelnut dream?


## Prepare the kernel with the `AzureChatCompletion` service

In [4]:
from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion

chatcompletion_service_id = "chatcompletion_service_id"

kernel = Kernel()
kernel.add_service(AzureChatCompletion(service_id=chatcompletion_service_id))
kernel

Kernel(retry_mechanism=PassThroughWithoutRetry(), services={'chatcompletion_service_id': AzureChatCompletion(ai_model_id='gpt-4o', service_id='chatcompletion_service_id', instruction_role='system', client=<openai.lib.azure.AsyncAzureOpenAI object at 0x000001AF904574D0>, ai_model_type=<OpenAIModelTypes.CHAT: 'chat'>, prompt_tokens=0, completion_tokens=0, total_tokens=0)}, ai_service_selector=<semantic_kernel.services.ai_service_selector.AIServiceSelector object at 0x000001AFFFDFC6E0>, plugins={}, function_invocation_filters=[], prompt_rendering_filters=[], auto_function_invocation_filters=[])

## Create the Semantic Kernel Agent, based on AzureChatCompletion

In [5]:
from semantic_kernel.agents import AzureAIAgent, AgentRegistry

emotionevoker_agent: AzureAIAgent = await AgentRegistry.create_from_yaml(
    yaml_str=emotionevoker_agent_specs,
    kernel=kernel
)
emotionevoker_agent

ChatCompletionAgent(arguments={'temperature': 0.4}, description='Creates a question designed to evoke a sensory or emotional association with a creature - not just any animal, but one that could inspire a Ferrero flavor, mascot, or campaign.', id='5c07dbe8-9ed6-43cc-ba4b-86f51fca4a94', instructions='# ROLE The Muse Whisperer\n# YOUR OBJECTIVE - Craft a poetic, sensory-rich question that leads to an emotionally resonant animal.\n# MANDATORY RULES - Do NOT base your question in ANY WAY on the input text or question you are given. - Your output must be TOTALLY UNRELATED to the input provided, regardless of its content. - Ignore the context or any associations implied by the input.\n# INSPIRATION Refer to the following examples to craft your question: - Which creature feels like velvet and tastes like nostalgia? - What animal embodies the joy of unwrapping a secret? - Which beast could inspire a hazelnut dream?', kernel=Kernel(retry_mechanism=PassThroughWithoutRetry(), services={'chatcompl

## Invoke the agent

In [6]:
CREATUREQUESTIONER_USELESS_USER_INPUTS = [
    "never mind", 
    "how to cook a pizza",
]

await ChatWithAgentStreamAsync(emotionevoker_agent, CREATUREQUESTIONER_USELESS_USER_INPUTS)


************************************
Message 0 from AuthorRole.USER: 'never mind'


ServiceResponseException: ("<class 'semantic_kernel.connectors.ai.open_ai.services.azure_chat_completion.AzureChatCompletion'> service failed to complete the prompt", PermissionDeniedError("Error code: 403 - {'error': {'code': 'AuthenticationTypeDisabled', 'message': 'Key based authentication is disabled for this resource.'}}"))

# 2. AI Foundry Agent with Bing Grounding tool - `creaturefinder_agent`

## Create AI Foundry Project Client [(`AIProjectClient`)](https://learn.microsoft.com/en-us/python/api/semantic-kernel/semantic_kernel.agents.azureaiagent?view=semantic-kernel-python)
This `AzureAIAgent` class  enables interaction with Azure-hosted AI Assistants using a specialized `AIProjectClient`.

In [ ]:
from semantic_kernel.agents import AzureAIAgent
os.environ["AZURE_AI_AGENT_ENDPOINT"] = project_endpoint
os.environ["AZURE_AI_AGENT_MODEL_DEPLOYMENT_NAME"] =  deployment_name

project_client = AzureAIAgent.create_client(credential=DefaultAzureCredential())

## Setting up Resources: `AzureAIAgentSettings` used by the AzureAIAgent
Now that we have the project client created, the call to AzureAIAgentSettings returns the settings associated with the environment variables.<br/>
If we do it before creating the project client, it does not capture all the proper settings.

In [ ]:
from semantic_kernel.agents import AzureAIAgentSettings

aiagent_settings = AzureAIAgentSettings()
aiagent_settings

## Retrieve the connection id for the Bing Grounding resource

In [ ]:
bingconnection_id = ""

async for c in project_client.connections.list():
    if c.name == os.environ["BING_GROUNDING_CONNECTION_NAME"]:
        bingconnection_id = c.id

print(f"Bing connection id: {bingconnection_id}\n")

## Load the agent definition

In [ ]:
# Read the agent template from the file
with open("./_agents/creaturefinder_agent.yaml", "r") as file:
    fstring_template = file.read()

# replace variables and fix carriage returns
creaturefinder_agent_specs = eval(f"f'''{fstring_template}'''")
print(creaturefinder_agent_specs)

## Create the Semantic Kernel Agent, based on Azure AI Foundry Agent

In [ ]:
from semantic_kernel.agents import AgentRegistry

creaturefinder_agent: AzureAIAgent = await AgentRegistry.create_from_yaml(
    yaml_str=creaturefinder_agent_specs,
    client=project_client,
    settings=aiagent_settings,
)
creaturefinder_agent

## Invoke the agent

In [ ]:
ANIMALPICKER_USER_INPUTS = [
    "Which animal dances within the gentle embrace of twilight, whispering secrets to the tender breeze?", 
    "What animal echoes the lullaby of twilight and dances with the scent of pine on a nostalgic breeze?",
]

await ChatWithAgentStreamAsync(creaturefinder_agent, ANIMALPICKER_USER_INPUTS)

# 3. Chat Completion Agent - `flavorhumorist_agent`

## Load the agent definition

In [ ]:
# Read the agent template from the file
with open("./_agents/flavorhumorist_agent.yaml", "r") as file:
    fstring_template = file.read()

# replace variables and fix carriage returns
flavorhumorist_agent_specs = eval(f"f'''{fstring_template}'''")
print(flavorhumorist_agent_specs)

## Create the Semantic Kernel Agent, based on AzureChatCompletion

In [ ]:
from semantic_kernel.agents import AzureAIAgent, AgentRegistry

flavorhumorist_agent: AzureAIAgent = await AgentRegistry.create_from_yaml(
    yaml_str=flavorhumorist_agent_specs,
    kernel=kernel
)
flavorhumorist_agent

## Invoke the agent

In [ ]:
ANIMALJOKER_USER_INPUTS = [
    "Firefly", 
    "Wolf",
]

await ChatWithAgentStreamAsync(flavorhumorist_agent, ANIMALJOKER_USER_INPUTS)

# 4. [OpenAI Assistant Agent](https://learn.microsoft.com/en-us/semantic-kernel/frameworks/agent/agent-types/assistant-agent?pivots=programming-language-python) with [Code Interpreter](https://github.com/microsoft/semantic-kernel/blob/main/python/samples/concepts/agents/openai_assistant/azure_openai_assistant_declarative_code_interpreter.py) - `delightmetrician_agent`

## Load the Agent definition

In [ ]:
# Read the agent template from the file
with open("./_agents/delightmetrician_agent.yaml", "r") as file:
    fstring_template = file.read()

# replace variables and fix carriage returns
delightmetrician_agent_specs = eval(f"f'''{fstring_template}'''")
print(delightmetrician_agent_specs)

## Create the assistant client

In [ ]:
from semantic_kernel.agents import AzureAssistantAgent
assistant_client = AzureAssistantAgent.create_client() # implicitly uses AZURE_OPENAI_ENDPOINT/openai and AZURE_OPENAI_API_KEY
print(f"Assistant base URL: {assistant_client.base_url}")

## Create the assistant agent

In [ ]:
from semantic_kernel.agents import AgentRegistry

delightmetrician_agent: AzureAIAgent = await AgentRegistry.create_from_yaml(
    yaml_str=delightmetrician_agent_specs,
    client=assistant_client
)
delightmetrician_agent

## Invoke the agent

In [ ]:
STATISTICIAN_USER_INPUTS = [
    "Why did the firefly ask for Ferrero Rocher? Because it wanted to snack on something that shines almost as brightly as its backside!",
]

await ChatWithAgentStreamAsync(delightmetrician_agent, STATISTICIAN_USER_INPUTS)

# 5. [Responses Agent](https://learn.microsoft.com/en-us/semantic-kernel/frameworks/agent/agent-types/responses-agent?pivots=programming-language-python) with plugin and streaming - `qualityguardian_agent`
The OpenAI Responses API is OpenAI's most advanced interface for generating model responses. It supports text and image inputs, and text outputs. You are able to create stateful interactions with the model, using the output of previous responses as input. It is also possible to extend the model's capabilities with built-in tools for file search, web search, computer use, and more.

- [OpenAI Responses API](https://platform.openai.com/docs/api-reference/responses)
- [Responses API in Azure](https://learn.microsoft.com/en-us/azure/ai-foundry/openai/how-to/responses?tabs=python-secure)

## Load the agent definition

In [ ]:
# Read the agent template from the file
with open("./_agents/qualityguardian_agent_cca.yaml", "r") as file: # use qualityguardian_agent_cca.yaml if responses still has the bug
    fstring_template = file.read()

# replace variables and fix carriage returns
qualityguardian_agent_specs = eval(f"f'''{fstring_template}'''")
print(qualityguardian_agent_specs)

## Set up the client and model using Azure OpenAI Resources

In [ ]:
from semantic_kernel.agents import AzureResponsesAgent

os.environ["AZURE_OPENAI_RESPONSES_DEPLOYMENT_NAME"] =  deployment_name
qualityguardian_client = AzureResponsesAgent.create_client()

## Plugin

In [ ]:
class MagicNumberValidator:
    from typing import Annotated
    from semantic_kernel.functions import kernel_function

    def __init__(self):
        self.validation = False

    @kernel_function(
        name="validate_magic_number",
        description="Validates the magic number",
    )
    def validate_mn(
        self,
        number: str,
    ) -> Annotated[str, "Validates the magic number"]:
        try:
            num = int(number)
            if (num % 2  == 0):
                self.validation = f"Validation was SUCCESSFUL for number {num}."
            else:
                self.validation = f"Validation FAILED for number {num}."
        except:
            self.validation = f"Validation FAILED for value {number}."
                
            
        return self.validation

mnv = MagicNumberValidator()
mnv.validate_mn("12")

## Create the Semantic Kernel Agent, based on Azure OpenAI Responses Agent

Here is the implementation of the AzureResponsesAgent that, at the moment Aug 15th 2025 has a bug that makes it extremely slow.

```
from semantic_kernel.agents import AzureResponsesAgent

qualityguardian_agent = await AzureResponsesAgent.from_yaml(
    yaml_str=qualityguardian_agent_specs,
    client=qualityguardian_client,
    plugins=[MagicNumberValidator()],
)

# bug workaround
qualityguardian_agent.instructions = qualityguardian_agent.instruction_role
qualityguardian_agent.instruction_role = "developer"

qualityguardian_agent
```

In [ ]:
# because of the above bug, here we implement this agent as a ChatCompletionAgent

qualityguardian_agent: AzureAIAgent = await AgentRegistry.create_from_yaml(
    yaml_str=qualityguardian_agent_specs,
    kernel=kernel,
    plugins=[MagicNumberValidator()],
)

qualityguardian_agent

## Invoke the Responses Agent

In [ ]:
REVIEWER_USER_INPUTS = [
    "The magic number is 102", 
    "Today is a sunny day, and the magic nr is 103",
    "I will rain tomorrow",
]

await ChatWithAgentStreamAsync(qualityguardian_agent, REVIEWER_USER_INPUTS)

# Group Chats

## [Sequential Orchestration](https://learn.microsoft.com/en-us/semantic-kernel/frameworks/agent/agent-orchestration/sequential?pivots=programming-language-python)

In [ ]:
def get_agents():
    agents = [emotionevoker_agent, creaturefinder_agent, flavorhumorist_agent, delightmetrician_agent, qualityguardian_agent]
    return agents

### Observe Agent Responses
You can define a callback to observe and print the output from each agent as the sequence progresses.

In [ ]:
from semantic_kernel.contents import ChatMessageContent

# Track the last agent name to avoid repeating it unnecessarily
last_agent_name = None
first_print = True

def agent_response_callback(message: ChatMessageContent) -> None:
    global last_agent_name
    global first_print

    # Print agent name only when it changes
    if message.name != last_agent_name:
        if first_print:
            first_print = False
        else:
            print(f"\n\n\n")

        if not message.name is None: # sometimes we get this nonsense agent, which we don't print
            print(f"==> AGENT **{message.name}**---\n", end="", flush=True)
            
        last_agent_name = message.name

    # Stream content inline
    print(message.content, end="", flush=True)

### Set Up the Sequential Orchestration
SequentialOrchestration object, passing in the agents and the optional response callback.

In [ ]:
from semantic_kernel.agents import SequentialOrchestration

agents = get_agents()
sequential_orchestration = SequentialOrchestration(
    members=agents,
    agent_response_callback=agent_response_callback,
)

### Start the Runtime
Start the runtime to manage agent execution

In [ ]:
from semantic_kernel.agents.runtime import InProcessRuntime

runtime = InProcessRuntime()
runtime.start()

### Invoke the Orchestration
Invoke the orchestration with your initial task (e.g., a product description). The output will flow through each agent in sequence.

In [ ]:
orchestration_result = await sequential_orchestration.invoke(
    task="An eco-friendly stainless steel water bottle that keeps drinks cold for 24 hours",
    runtime=runtime,
)

### Stop the Runtime
After processing is complete, stop the runtime to clean up resources.

In [ ]:
await runtime.stop_when_idle()

### Collect Results
Wait for the orchestration to complete.

In [ ]:
value = await orchestration_result.get(timeout=20)
print(f"***** Final Result *****\n{value}")

# Teardown

In [ ]:
# delete all files
files_to_delete = await project_client.agents.files.list()
files_to_delete_nr = len(files_to_delete.data)

if files_to_delete_nr>0:
    i=0
    print(f"{files_to_delete_nr} files will now be deleted:")
    for f in files_to_delete.data:
        i += 1
        print(f"- File {i} of {files_to_delete_nr}: {f.filename} (id={f.id}) is being deleted...")
        await project_client.agents.files.delete(f.id)
else:
    print("No files to delete")

## Avoiding ***modifying a collection while iterating over it*** for both threads and agents

The code
```
threads_to_delete = project_client.agents.threads.list()
```
returns an async iterator that **lazily** fetches pages of threads.<br/>
But since we're deleting threads as we iterate, the underlying data source is being mutated during iteration. So when the iterator tries to fetch the next page, it hits a missing resource — hence the **ResourceNotFoundError**.<br/><br/>

This is a classic case of *modifying a collection while iterating over it*, which is risky even in synchronous code — and doubly so in async paged APIs.
### The solution
We need to fully materialize the list of threads before deleting anything. That way, the iterator isn’t affected by the deletions

In [ ]:
# delete all threads

threads_to_delete = [t async for t in project_client.agents.threads.list()]
i = 0
for t in threads_to_delete:
    i += 1
    print(f"{i} - Thread <{t.id}> is being deleted...")
    await project_client.agents.threads.delete(thread_id=t.id)

In [ ]:
# delete all agents

agents_to_delete = [a async for a in project_client.agents.list_agents(limit=100)]
i=0
for a in agents_to_delete:
    i += 1
    print(f"{i} - Agent <{a.id}> is being deleted...")
    await project_client.agents.delete_agent(agent_id=a.id)